In [ ]:
#***************************************************************************************************************************
# DEV ENVIRONMENT SETUP
#***************************************************************************************************************************

In [ ]:
#=====================================================
# Install libraries
#=====================================================

In [282]:
!pip install pandas requests duckdb openpyxl matplotlib seaborn numpy

In [283]:
#=====================================================
# Import libraries
#=====================================================

In [284]:
import pandas as pd
import requests
import duckdb
import zipfile
import os

In [285]:
#=====================================================
# Generic functions
#=====================================================

In [286]:
# function to download dataset into browser
def download(url, filename):
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        with open(filename, "wb") as f:
            f.write(response.content)
            
        print("Download successful:", filename)
        
    except Exception as e:
        print("Error:", e)

# function to download zipped dataset into browser and extract files in a specified folder
def download_and_extract_zip(url, zip_path, extract_path):
    try:
        # download
        response = requests.get(url)
        with open(zip_path, "wb") as f:
            f.write(response.content)
        
        # extract
        os.makedirs(extract_path, exist_ok=True)
        
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
            
    except Exception as e:
        print("Error:", e)

    print("Download and extraction complete")

In [379]:
def move_columns(df, cols_to_move, after=None, before=None):
    """
    Déplace une ou plusieurs colonnes dans un DataFrame.

    Parameters:
    - df : pandas DataFrame
    - cols_to_move : list de colonnes à déplacer
    - after : colonne après laquelle insérer
    - before : colonne avant laquelle insérer

    Returns:
    - DataFrame avec colonnes réordonnées
    """

    cols = list(df.columns)

    # sécuriser : transformer en liste si une seule colonne
    if isinstance(cols_to_move, str):
        cols_to_move = [cols_to_move]

    # enlever les colonnes à déplacer
    for col in cols_to_move:
        if col in cols:
            cols.remove(col)

    # déterminer position
    if after:
        pos = cols.index(after) + 1
    elif before:
        pos = cols.index(before)
    else:
        pos = len(cols)  # à la fin

    # insérer les colonnes
    cols[pos:pos] = cols_to_move

    return df[cols]

In [287]:
#***************************************************************************************************************************
# QUALITY OF LIFE IN OUR CITIES / QUALITE DE VIE DANS NOS CITEES
#***************************************************************************************************************************

In [288]:
#=====================================================
# I - STANDARD OF LIVING / NIVEAU DE VIE
#=====================================================

In [289]:
#-----------------------------------------------------
# I.1 - INCOME analysis / Analyse des SALAIRES
#-----------------------------------------------------

In [290]:
# A - Download zip file and extract datasets
# A.1 - Declared Incomes
url1 = "https://www.insee.fr/fr/statistiques/fichier/8229323/BASE_TD_FILO_IRIS_2021_DEC_CSV.zip"
download_and_extract_zip(url1, "BASE_TD_FILO_IRIS_2021_DEC_CSV.zip", "Standard_of_living/Income/Declared_income_2021")
# A.2 - Disposable Incomes
url2 = "https://www.insee.fr/fr/statistiques/fichier/8229323/BASE_TD_FILO_IRIS_2021_DISP_CSV.zip"
download_and_extract_zip(url2, "BASE_TD_FILO_IRIS_2021_DISP_CSV.zip", "Standard_of_living/Income/Disposable_income_2021")

Download and extraction complete
Download and extraction complete


In [291]:
# A.3 - INSEE codes for locations
url3 = "https://www.insee.fr/fr/statistiques/fichier/8377162/cog_ensemble_2025_csv.zip"
download_and_extract_zip(url3, "cog_ensemble_2025_csv.zip", "Standard_of_living")

Download and extraction complete


In [292]:
# B.1 - Read dataset Declared Incomes
declared_income_2021 = pd.read_csv("Standard_of_living/Income/Declared_income_2021/BASE_TD_FILO_IRIS_2021_DEC.csv", sep=";")
declared_income_2021.head()

,IRIS,DEC_PIMP21,DEC_TP6021,DEC_INCERT21,DEC_Q121,DEC_MED21,DEC_Q321,DEC_EQ21,DEC_D121,DEC_D221,...,DEC_RD21,DEC_S80S2021,DEC_GI21,DEC_PACT21,DEC_PTSA21,DEC_PCHO21,DEC_PBEN21,DEC_PPEN21,DEC_PAUT21,DEC_NOTE21
0,010040101,"43,0","29,0",2,12610,19330,26390,"0,71",7760,11300,...,"4,4","6,3","0,318","69,3","62,5","3,2","3,6","27,1","3,6",0
1,010040102,"42,0","39,0",1,9730,16830,24620,"0,89",4680,8600,...,"7,1","9,0","0,362","70,5","63,7","4,4","2,4","25,6","3,9",0
2,010040201,"47,0","29,0",1,12220,19940,27650,"0,77",6500,10300,...,"5,7","8,0","0,352","69,3","61,9","3,6","3,8","26,7","4,0",0
3,010040202,"62,0","14,0",1,18350,25560,35010,"0,65",11960,16450,...,"3,8","6,3","0,360","65,5","60,0","2,3","3,2","21,8","12,7",0
4,010330102,"42,0","31,0",1,11280,19870,30050,"0,94",5250,9790,...,"8,5","10,6","0,390","74,1","67,5","4,8","1,8","23,0","2,9",0


In [293]:
declared_income_2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 16026 entries, 0 to 16025
Data columns (total 26 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   IRIS          16026 non-null  str  
 1   DEC_PIMP21    16026 non-null  str  
 2   DEC_TP6021    16026 non-null  str  
 3   DEC_INCERT21  16026 non-null  str  
 4   DEC_Q121      16026 non-null  str  
 5   DEC_MED21     16026 non-null  str  
 6   DEC_Q321      16026 non-null  str  
 7   DEC_EQ21      16026 non-null  str  
 8   DEC_D121      16026 non-null  str  
 9   DEC_D221      16026 non-null  str  
 10  DEC_D321      16026 non-null  str  
 11  DEC_D421      16026 non-null  str  
 12  DEC_D621      16026 non-null  str  
 13  DEC_D721      16026 non-null  str  
 14  DEC_D821      16026 non-null  str  
 15  DEC_D921      16026 non-null  str  
 16  DEC_RD21      16026 non-null  str  
 17  DEC_S80S2021  16026 non-null  str  
 18  DEC_GI21      16026 non-null  str  
 19  DEC_PACT21    16026 non-null  str  


In [294]:
# B.2 - Read dataset Disposable Incomes
disposable_income_2021 = pd.read_csv("Standard_of_living/Income/Disposable_income_2021/BASE_TD_FILO_IRIS_2021_DISP.csv", sep=";")
disposable_income_2021.head()

,IRIS,DISP_TP6021,DISP_INCERT21,DISP_Q121,DISP_MED21,DISP_Q321,DISP_EQ21,DISP_D121,DISP_D221,DISP_D321,...,DISP_PCHO21,DISP_PBEN21,DISP_PPEN21,DISP_PPAT21,DISP_PPSOC21,DISP_PPFAM21,DISP_PPMINI21,DISP_PPLOGT21,DISP_PIMPOT21,DISP_NOTE21
0,010040101,"19,0",2,14990,20350,26140,"0,55",11620,14280,16080,...,"3,0","3,6","26,9","6,2","8,6","3,3","3,8","1,5","-12,5",0
1,010040102,"25,0",1,13880,18570,24760,"0,59",10580,12890,14660,...,"4,2","2,4","24,9","5,8","11,1","3,7","5,1","2,3","-12,4",0
2,010040201,"19,0",1,15190,20700,27180,"0,58",11400,14060,16320,...,"3,5","4,0","27,2","6,4","7,7","2,8","3,3","1,6","-13,8",0
3,010040202,"8,0",1,19600,25230,33170,"0,54",14810,18310,20780,...,"2,4","3,6","23,8","16,2","4,0","1,8","1,5","0,7","-17,3",0
4,010330102,"24,0",1,14050,20420,29640,"0,76",9410,12570,15130,...,"4,9","1,9","23,7","5,2","5,3","1,5","2,5","1,3","-13,1",0


In [295]:
disposable_income_2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 16026 entries, 0 to 16025
Data columns (total 30 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   IRIS           16026 non-null  str  
 1   DISP_TP6021    16026 non-null  str  
 2   DISP_INCERT21  16026 non-null  str  
 3   DISP_Q121      16026 non-null  str  
 4   DISP_MED21     16026 non-null  str  
 5   DISP_Q321      16026 non-null  str  
 6   DISP_EQ21      16026 non-null  str  
 7   DISP_D121      16026 non-null  str  
 8   DISP_D221      16026 non-null  str  
 9   DISP_D321      16026 non-null  str  
 10  DISP_D421      16026 non-null  str  
 11  DISP_D621      16026 non-null  str  
 12  DISP_D721      16026 non-null  str  
 13  DISP_D821      16026 non-null  str  
 14  DISP_D921      16026 non-null  str  
 15  DISP_RD21      16026 non-null  str  
 16  DISP_S80S2021  16026 non-null  str  
 17  DISP_GI21      16026 non-null  str  
 18  DISP_PACT21    16026 non-null  str  
 19  DISP_PTSA21    

In [296]:
# B.3 - Read dataset INSEE code to commune name

In [297]:
insee_to_commune = pd.read_csv("Standard_of_living/v_commune_2025.csv", sep=",")
insee_to_commune.head() 

,TYPECOM,COM,REG,DEP,CTCD,ARR,TNCC,NCC,NCCENR,LIBELLE,CAN,COMPARENT
0,COM,01001,84.0,01,01D,012,5,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,L'Abergement-Clémenciat,0108,NaN
1,COM,01002,84.0,01,01D,011,5,ABERGEMENT DE VAREY,Abergement-de-Varey,L'Abergement-de-Varey,0101,NaN
2,COM,01004,84.0,01,01D,011,1,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,Ambérieu-en-Bugey,0101,NaN
3,COM,01005,84.0,01,01D,012,1,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,Ambérieux-en-Dombes,0122,NaN
4,COM,01006,84.0,01,01D,011,1,AMBLEON,Ambléon,Ambléon,0104,NaN


In [298]:
# Only keep the real commune names: filter on "TYPECOM" = "COM"
insee_to_commune = insee_to_commune[
    insee_to_commune["TYPECOM"] == "COM"
].copy()

In [299]:
# B.4 - Read dataset IRIS code to precise location in commune
iris_to_commune = pd.read_csv("Standard_of_living/Income/Disposable_income_2021/meta_BASE_TD_FILO_IRIS_2021_DISP.csv", sep=";")
iris_to_commune.head() 

,COD_VAR,LIB_VAR,LIB_VAR_LONG,COD_MOD,LIB_MOD,TYPE_VAR,LONG_VAR
0,IRIS,IRIS,Code du département suivi du numéro de commune...,NaN,NaN,CHAR,9
1,DISP_TP6021,Taux de pauvreté au seuil de 60 % (%),Taux de pauvreté au seuil de 60 % du revenu di...,NaN,NaN,CHAR,4
2,DISP_INCERT21,Incertitude sur les indicateurs DISP_TP6021,Incertitude sur le taux de bas revenus,1,IRIS de 1 000 à moins de 10 000 ménages : à l’...,CHAR,2
3,DISP_INCERT21,Incertitude sur les indicateurs DISP_TP6021,Incertitude sur le taux de bas revenus,2,IRIS de 500 à moins de 1 000 ménages : à l’ent...,CHAR,2
4,DISP_Q121,1ᵉʳ quartile (€),1ᵉʳ quartile du revenu disponible par unité de...,NaN,NaN,CHAR,5


In [300]:
# C - Clean dataset 

In [301]:
#C.1.1 declared_income_2021 - Keep only needed columns

In [302]:
declared_income_2021.drop(
    columns=[
        "DEC_INCERT21","DEC_D221","DEC_D321","DEC_D421","DEC_D621","DEC_D721","DEC_D821",
        "DEC_EQ21","DEC_RD21","DEC_S80S2021","DEC_GI21",
        "DEC_PTSA21","DEC_PCHO21","DEC_PBEN21","DEC_NOTE21"
    ],
    inplace=True
)
declared_income_2021.head()

,IRIS,DEC_PIMP21,DEC_TP6021,DEC_Q121,DEC_MED21,DEC_Q321,DEC_D121,DEC_D921,DEC_PACT21,DEC_PPEN21,DEC_PAUT21
0,010040101,"43,0","29,0",12610,19330,26390,7760,33850,"69,3","27,1","3,6"
1,010040102,"42,0","39,0",9730,16830,24620,4680,33030,"70,5","25,6","3,9"
2,010040201,"47,0","29,0",12220,19940,27650,6500,37220,"69,3","26,7","4,0"
3,010040202,"62,0","14,0",18350,25560,35010,11960,45520,"65,5","21,8","12,7"
4,010330102,"42,0","31,0",11280,19870,30050,5250,44820,"74,1","23,0","2,9"


In [303]:
#C.2.1 disposable_income_2021 - Keep only needed columns

In [304]:
disposable_income_2021.drop(
    columns=[
        "DISP_INCERT21","DISP_D221","DISP_D321","DISP_D421","DISP_D621","DISP_D721","DISP_D821",
        "DISP_EQ21","DISP_RD21","DISP_S80S2021","DISP_GI21",
        "DISP_PTSA21","DISP_PCHO21","DISP_PBEN21","DISP_NOTE21"
    ],
    inplace=True
)
disposable_income_2021.head()

,IRIS,DISP_TP6021,DISP_Q121,DISP_MED21,DISP_Q321,DISP_D121,DISP_D921,DISP_PACT21,DISP_PPEN21,DISP_PPAT21,DISP_PPSOC21,DISP_PPFAM21,DISP_PPMINI21,DISP_PPLOGT21,DISP_PIMPOT21
0,010040101,"19,0",14990,20350,26140,11620,32060,"70,8","26,9","6,2","8,6","3,3","3,8","1,5","-12,5"
1,010040102,"25,0",13880,18570,24760,10580,31130,"70,6","24,9","5,8","11,1","3,7","5,1","2,3","-12,4"
2,010040201,"19,0",15190,20700,27180,11400,34450,"72,5","27,2","6,4","7,7","2,8","3,3","1,6","-13,8"
3,010040202,"8,0",19600,25230,33170,14810,41230,"73,3","23,8","16,2","4,0","1,8","1,5","0,7","-17,3"
4,010330102,"24,0",14050,20420,29640,9410,42390,"78,9","23,7","5,2","5,3","1,5","2,5","1,3","-13,1"


In [305]:
#C.3.1 insee_to_commune - Keep only needed columns

In [306]:
insee_to_commune.drop(
    columns=[
        "TYPECOM","REG","DEP","CTCD","ARR","TNCC","LIBELLE","CAN","COMPARENT"
    ],
    inplace=True
)
insee_to_commune.head()

,COM,NCC,NCCENR
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes
4,01006,AMBLEON,Ambléon


In [307]:
#C.4.1 iris_to_commune - Keep only needed columns

In [308]:
iris_to_commune.drop(
    columns=[
        "COD_VAR","LIB_VAR","LIB_VAR_LONG","TYPE_VAR","LONG_VAR"
    ],
    inplace=True
)
iris_to_commune.head()

,COD_MOD,LIB_MOD
0,NaN,NaN
1,NaN,NaN
2,1,IRIS de 1 000 à moins de 10 000 ménages : à l’...
3,2,IRIS de 500 à moins de 1 000 ménages : à l’ent...
4,NaN,NaN


In [309]:
iris_to_commune = iris_to_commune.dropna()
iris_to_commune.head(10)

,COD_MOD,LIB_MOD
2,1,IRIS de 1 000 à moins de 10 000 ménages : à l’...
3,2,IRIS de 500 à moins de 1 000 ménages : à l’ent...
30,0,Aucun problème particulier. Certaines données ...
31,1,IRIS pour lesquels l’évolution du nombre de mé...
32,5,Données non diffusées en raison d'anomalies re...
33,010040101,Les Pérouses-Triangle d'Activités
34,010040102,Longeray-Gare
35,010040201,Centre-Saint-Germain-Vareilles
36,010040202,Tiret-Les Allymes
37,010330102,Centre Ville


In [310]:
iris_to_commune = iris_to_commune[iris_to_commune.index >= 33]
iris_to_commune.reset_index(drop=True, inplace=True)
iris_to_commune.head()

,COD_MOD,LIB_MOD
0,010040101,Les Pérouses-Triangle d'Activités
1,010040102,Longeray-Gare
2,010040201,Centre-Saint-Germain-Vareilles
3,010040202,Tiret-Les Allymes
4,010330102,Centre Ville


In [311]:
#C.1.2 declared_income_2021 - Rename columns

In [312]:
rename_dict = {
    "IRIS": "iris_id",
    "DEC_PIMP21": "menages_fiscaux_imposes_pct",
    "DEC_TP6021": "taux_bas_revenus_pct",
    "DEC_Q121": "revenu_q1_eur",
    "DEC_MED21": "revenu_median_eur",
    "DEC_Q321": "revenu_q3_eur",
    "DEC_D121": "revenu_d1_eur",
    "DEC_D921": "revenu_d9_eur",
    "DEC_PACT21": "part_revenus_activite_pct",
    "DEC_PPEN21": "part_pensions_pct",
    "DEC_PAUT21": "part_autres_revenus_pct"
}
declared_income_2021.rename(
    columns=rename_dict,
    inplace=True
)
declared_income_2021.head()

,iris_id,menages_fiscaux_imposes_pct,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_autres_revenus_pct
0,010040101,"43,0","29,0",12610,19330,26390,7760,33850,"69,3","27,1","3,6"
1,010040102,"42,0","39,0",9730,16830,24620,4680,33030,"70,5","25,6","3,9"
2,010040201,"47,0","29,0",12220,19940,27650,6500,37220,"69,3","26,7","4,0"
3,010040202,"62,0","14,0",18350,25560,35010,11960,45520,"65,5","21,8","12,7"
4,010330102,"42,0","31,0",11280,19870,30050,5250,44820,"74,1","23,0","2,9"


In [313]:
#C.2.2 disposable_income_2021 - Rename columns

In [314]:
rename_dict = {
    "IRIS": "iris_id",
    "DISP_TP6021": "taux_bas_revenus_pct",
    "DISP_Q121": "revenu_q1_eur",
    "DISP_MED21": "revenu_median_eur",
    "DISP_Q321": "revenu_q3_eur",
    "DISP_D121": "revenu_d1_eur",
    "DISP_D921": "revenu_d9_eur",
    "DISP_PACT21": "part_revenus_activite_pct",
    "DISP_PPEN21": "part_pensions_pct",
    "DISP_PPAT21": "part_revenus_patrimoine_pct",
    "DISP_PPSOC21": "part_prestations_sociales_pct",
    "DISP_PPFAM21": "part_prestations_familiales_pct",
    "DISP_PPMINI21": "part_minima_sociaux_pct",
    "DISP_PPLOGT21": "part_prestations_logement_pct",
    "DISP_PIMPOT21": "part_impots_pct"
}
disposable_income_2021.rename(
    columns=rename_dict,
    inplace=True
)
disposable_income_2021.head()

,iris_id,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_revenus_patrimoine_pct,part_prestations_sociales_pct,part_prestations_familiales_pct,part_minima_sociaux_pct,part_prestations_logement_pct,part_impots_pct
0,010040101,"19,0",14990,20350,26140,11620,32060,"70,8","26,9","6,2","8,6","3,3","3,8","1,5","-12,5"
1,010040102,"25,0",13880,18570,24760,10580,31130,"70,6","24,9","5,8","11,1","3,7","5,1","2,3","-12,4"
2,010040201,"19,0",15190,20700,27180,11400,34450,"72,5","27,2","6,4","7,7","2,8","3,3","1,6","-13,8"
3,010040202,"8,0",19600,25230,33170,14810,41230,"73,3","23,8","16,2","4,0","1,8","1,5","0,7","-17,3"
4,010330102,"24,0",14050,20420,29640,9410,42390,"78,9","23,7","5,2","5,3","1,5","2,5","1,3","-13,1"


In [315]:
#C.3.2 insee_to_commune - Rename columns

In [316]:
rename_dict = {
    "COM": "insee_commune_id",
    "NCC": "commune_name_upper",
    "NCCENR": "commune_name"
}
insee_to_commune.rename(
    columns=rename_dict,
    inplace=True
)
insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes
4,01006,AMBLEON,Ambléon


In [317]:
#C.4.2 iris_to_commune - Rename columns

In [318]:
rename_dict = {
    "COD_MOD": "iris_id",
    "LIB_MOD": "place_in_commune"
}
iris_to_commune.rename(
    columns=rename_dict,
    inplace=True
)
iris_to_commune.head()

,iris_id,place_in_commune
0,010040101,Les Pérouses-Triangle d'Activités
1,010040102,Longeray-Gare
2,010040201,Centre-Saint-Germain-Vareilles
3,010040202,Tiret-Les Allymes
4,010330102,Centre Ville


In [319]:
#C.4.1.3 declared_income_2021 - CAST numeric data appearing as STR 

In [320]:
declared_income_2021.dtypes

iris_id                        str
menages_fiscaux_imposes_pct    str
taux_bas_revenus_pct           str
revenu_q1_eur                  str
revenu_median_eur              str
revenu_q3_eur                  str
revenu_d1_eur                  str
revenu_d9_eur                  str
part_revenus_activite_pct      str
part_pensions_pct              str
part_autres_revenus_pct        str
dtype: object

In [321]:
for col in declared_income_2021.columns:
    if col != "iris_id":
        declared_income_2021[col] = pd.to_numeric(declared_income_2021[col].str.replace(",", ".", regex=False), errors="coerce")
        # errors="coerce" ==> when conversion fails, change value in NaN
declared_income_2021.head()

,iris_id,menages_fiscaux_imposes_pct,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_autres_revenus_pct
0,010040101,43.0,29.0,12610.0,19330.0,26390.0,7760.0,33850.0,69.3,27.1,3.6
1,010040102,42.0,39.0,9730.0,16830.0,24620.0,4680.0,33030.0,70.5,25.6,3.9
2,010040201,47.0,29.0,12220.0,19940.0,27650.0,6500.0,37220.0,69.3,26.7,4.0
3,010040202,62.0,14.0,18350.0,25560.0,35010.0,11960.0,45520.0,65.5,21.8,12.7
4,010330102,42.0,31.0,11280.0,19870.0,30050.0,5250.0,44820.0,74.1,23.0,2.9


In [322]:
declared_income_2021.dtypes

iris_id                            str
menages_fiscaux_imposes_pct    float64
taux_bas_revenus_pct           float64
revenu_q1_eur                  float64
revenu_median_eur              float64
revenu_q3_eur                  float64
revenu_d1_eur                  float64
revenu_d9_eur                  float64
part_revenus_activite_pct      float64
part_pensions_pct              float64
part_autres_revenus_pct        float64
dtype: object

In [323]:
#C.4.2.3 disposable_income_2021 - CAST numeric data appearing as STR 

In [324]:
for col in disposable_income_2021.columns:
    if col != "iris_id":
        disposable_income_2021[col] = pd.to_numeric(disposable_income_2021[col].str.replace(",", ".", regex=False), errors="coerce")
disposable_income_2021.dtypes

iris_id                                str
taux_bas_revenus_pct               float64
revenu_q1_eur                      float64
revenu_median_eur                  float64
revenu_q3_eur                      float64
revenu_d1_eur                      float64
revenu_d9_eur                      float64
part_revenus_activite_pct          float64
part_pensions_pct                  float64
part_revenus_patrimoine_pct        float64
part_prestations_sociales_pct      float64
part_prestations_familiales_pct    float64
part_minima_sociaux_pct            float64
part_prestations_logement_pct      float64
part_impots_pct                    float64
dtype: object

In [325]:
#C.4.1.4 declared_income_2021 - Check if NaN values

In [326]:
declared_income_2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 16026 entries, 0 to 16025
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   iris_id                      16026 non-null  str    
 1   menages_fiscaux_imposes_pct  14490 non-null  float64
 2   taux_bas_revenus_pct         14489 non-null  float64
 3   revenu_q1_eur                14490 non-null  float64
 4   revenu_median_eur            14490 non-null  float64
 5   revenu_q3_eur                14490 non-null  float64
 6   revenu_d1_eur                14490 non-null  float64
 7   revenu_d9_eur                14490 non-null  float64
 8   part_revenus_activite_pct    14490 non-null  float64
 9   part_pensions_pct            14490 non-null  float64
 10  part_autres_revenus_pct      14490 non-null  float64
dtypes: float64(10), str(1)
memory usage: 1.3 MB


In [327]:
#Select empty lines (appart from IRIS code)
cols = [col for col in declared_income_2021.columns if col != "iris_id"]
null_rows_declared = declared_income_2021[declared_income_2021[cols].isna().all(axis=1)]
null_rows_declared.head(30)

,iris_id,menages_fiscaux_imposes_pct,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_autres_revenus_pct
59,012830302,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63,012830306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,021680301,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,024080102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92,024080103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,024080701,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,026910210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
127,026910505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
176,031850107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
178,031850109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [328]:
#C.4.2.4 disposable_income_2021 - Check if NaN values

In [329]:
disposable_income_2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 16026 entries, 0 to 16025
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   iris_id                          16026 non-null  str    
 1   taux_bas_revenus_pct             14486 non-null  float64
 2   revenu_q1_eur                    14490 non-null  float64
 3   revenu_median_eur                14490 non-null  float64
 4   revenu_q3_eur                    14490 non-null  float64
 5   revenu_d1_eur                    14490 non-null  float64
 6   revenu_d9_eur                    14490 non-null  float64
 7   part_revenus_activite_pct        14490 non-null  float64
 8   part_pensions_pct                14490 non-null  float64
 9   part_revenus_patrimoine_pct      14490 non-null  float64
 10  part_prestations_sociales_pct    14490 non-null  float64
 11  part_prestations_familiales_pct  14490 non-null  float64
 12  part_minima_sociaux_pct      

In [330]:
#Select empty lines (appart from IRIS code)
cols = [col for col in disposable_income_2021.columns if col != "iris_id"]
null_rows_disposable = disposable_income_2021[disposable_income_2021[cols].isna().all(axis=1)]
null_rows_disposable.head(30)

,iris_id,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_revenus_patrimoine_pct,part_prestations_sociales_pct,part_prestations_familiales_pct,part_minima_sociaux_pct,part_prestations_logement_pct,part_impots_pct
59,012830302,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63,012830306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,021680301,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,024080102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92,024080103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,024080701,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,026910210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
127,026910505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
176,031850107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
178,031850109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [331]:
null_rows_declared.info()

<class 'pandas.DataFrame'>
Index: 1536 entries, 59 to 16025
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   iris_id                      1536 non-null   str    
 1   menages_fiscaux_imposes_pct  0 non-null      float64
 2   taux_bas_revenus_pct         0 non-null      float64
 3   revenu_q1_eur                0 non-null      float64
 4   revenu_median_eur            0 non-null      float64
 5   revenu_q3_eur                0 non-null      float64
 6   revenu_d1_eur                0 non-null      float64
 7   revenu_d9_eur                0 non-null      float64
 8   part_revenus_activite_pct    0 non-null      float64
 9   part_pensions_pct            0 non-null      float64
 10  part_autres_revenus_pct      0 non-null      float64
dtypes: float64(10), str(1)
memory usage: 144.0 KB


In [332]:
null_rows_disposable.info()

<class 'pandas.DataFrame'>
Index: 1536 entries, 59 to 16025
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   iris_id                          1536 non-null   str    
 1   taux_bas_revenus_pct             0 non-null      float64
 2   revenu_q1_eur                    0 non-null      float64
 3   revenu_median_eur                0 non-null      float64
 4   revenu_q3_eur                    0 non-null      float64
 5   revenu_d1_eur                    0 non-null      float64
 6   revenu_d9_eur                    0 non-null      float64
 7   part_revenus_activite_pct        0 non-null      float64
 8   part_pensions_pct                0 non-null      float64
 9   part_revenus_patrimoine_pct      0 non-null      float64
 10  part_prestations_sociales_pct    0 non-null      float64
 11  part_prestations_familiales_pct  0 non-null      float64
 12  part_minima_sociaux_pct          0

In [333]:
#C.4.3.4 insee_to_commune - Check if NaN values

In [334]:
insee_to_commune.info()

<class 'pandas.DataFrame'>
Index: 34875 entries, 0 to 37547
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   insee_commune_id    34875 non-null  str  
 1   commune_name_upper  34875 non-null  str  
 2   commune_name        34875 non-null  str  
dtypes: str(3)
memory usage: 1.1 MB


In [335]:
#OK ==> No NULL values in insee_to_commune table

In [336]:
#C.4.4.4 iris_to_commune - Check if NaN values

In [337]:
iris_to_commune.info()

<class 'pandas.DataFrame'>
RangeIndex: 16026 entries, 0 to 16025
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   iris_id           16026 non-null  str  
 1   place_in_commune  16026 non-null  str  
dtypes: str(2)
memory usage: 250.5 KB


In [338]:
#OK ==> No NULL values in iris_to_commune table

In [339]:
#C.4.1.5 declared_income_2021 - Remove empty lines

In [340]:
declared_income_2021 = declared_income_2021.drop(index=null_rows_declared.index)

In [341]:
declared_income_2021.info()

<class 'pandas.DataFrame'>
Index: 14490 entries, 0 to 16022
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   iris_id                      14490 non-null  str    
 1   menages_fiscaux_imposes_pct  14490 non-null  float64
 2   taux_bas_revenus_pct         14489 non-null  float64
 3   revenu_q1_eur                14490 non-null  float64
 4   revenu_median_eur            14490 non-null  float64
 5   revenu_q3_eur                14490 non-null  float64
 6   revenu_d1_eur                14490 non-null  float64
 7   revenu_d9_eur                14490 non-null  float64
 8   part_revenus_activite_pct    14490 non-null  float64
 9   part_pensions_pct            14490 non-null  float64
 10  part_autres_revenus_pct      14490 non-null  float64
dtypes: float64(10), str(1)
memory usage: 1.3 MB


In [342]:
#C.4.2.5 declared_income_2021 - Remove empty lines

In [343]:
disposable_income_2021 = disposable_income_2021.drop(index=null_rows_disposable.index)

In [344]:
disposable_income_2021.info()

<class 'pandas.DataFrame'>
Index: 14490 entries, 0 to 16022
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   iris_id                          14490 non-null  str    
 1   taux_bas_revenus_pct             14486 non-null  float64
 2   revenu_q1_eur                    14490 non-null  float64
 3   revenu_median_eur                14490 non-null  float64
 4   revenu_q3_eur                    14490 non-null  float64
 5   revenu_d1_eur                    14490 non-null  float64
 6   revenu_d9_eur                    14490 non-null  float64
 7   part_revenus_activite_pct        14490 non-null  float64
 8   part_pensions_pct                14490 non-null  float64
 9   part_revenus_patrimoine_pct      14490 non-null  float64
 10  part_prestations_sociales_pct    14490 non-null  float64
 11  part_prestations_familiales_pct  14490 non-null  float64
 12  part_minima_sociaux_pct          1

In [345]:
#C.4.4.5 iris_to_commune - Flag communes with empty revenu data in iris_to_commune table

In [346]:
iris_with_data = set(declared_income_2021["iris_id"])
iris_to_commune["has_income_data"] = iris_to_commune["iris_id"].isin(iris_with_data)
iris_to_commune.head()

,iris_id,place_in_commune,has_income_data
0,010040101,Les Pérouses-Triangle d'Activités,True
1,010040102,Longeray-Gare,True
2,010040201,Centre-Saint-Germain-Vareilles,True
3,010040202,Tiret-Les Allymes,True
4,010330102,Centre Ville,True


In [347]:
#C.4.4.6 iris_to_commune - add a column insee_commune_id and fill it with the first 5 chars of IRIS CODE

In [348]:
pos = iris_to_commune.columns.get_loc("iris_id")
iris_to_commune.insert(loc=pos + 1,column="insee_commune_id",value=iris_to_commune["iris_id"].str[:5])

In [349]:
iris_to_commune.head()

,iris_id,insee_commune_id,place_in_commune,has_income_data
0,010040101,01004,Les Pérouses-Triangle d'Activités,True
1,010040102,01004,Longeray-Gare,True
2,010040201,01004,Centre-Saint-Germain-Vareilles,True
3,010040202,01004,Tiret-Les Allymes,True
4,010330102,01033,Centre Ville,True


In [350]:
pos = iris_to_commune.columns.get_loc("insee_commune_id")
iris_to_commune.insert(
             loc=pos + 1,
             column="commune_name",
             value=duckdb.query("""
                SELECT 
                    c.commune_name
                    FROM iris_to_commune i
                    LEFT JOIN (
                        SELECT insee_commune_id, commune_name
                        FROM insee_to_commune
                    ) c
                    ON i.insee_commune_id = c.insee_commune_id
                    ORDER BY i.iris_id
                """).df()
            )

In [351]:
iris_to_commune.head(30)

,iris_id,insee_commune_id,commune_name,place_in_commune,has_income_data
0,010040101,01004,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,True
1,010040102,01004,Ambérieu-en-Bugey,Longeray-Gare,True
2,010040201,01004,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,True
3,010040202,01004,Ambérieu-en-Bugey,Tiret-Les Allymes,True
4,010330102,01033,Valserhône,Centre Ville,True
5,010330103,01033,Valserhône,Lancrans-Coupy-Vanchy,True
6,010330201,01033,Valserhône,Arc Vouvray-Gare-Châtillon,True
7,010330202,01033,Valserhône,Plateau de Musinens,True
8,010330301,01033,Valserhône,Arlod,True
9,010330401,01033,Valserhône,Châtillon-en-Michaille,True


In [352]:
#D - Save all cleaned tables in csv formats in current folder

In [353]:
declared_income_2021.to_csv("declared_income_2021_clean.csv", index=False)
disposable_income_2021.to_csv("disposable_income_2021_clean.csv", index=False)
insee_to_commune.to_csv("insee_to_commune_clean.csv", index=False)
iris_to_commune.to_csv("iris_to_commune_clean.csv", index=False)

In [ ]:
#======================================================
# II - SECURITY / SECURITE
#======================================================

In [1]:
#------------------------------------------------------
# II.a - CRIME analysis / Analyse de DELINQUANCE
#------------------------------------------------------

In [ ]:
# A - Download compressed file 

In [32]:
url3 = "https://www.data.gouv.fr/api/1/datasets/r/6252a84c-6b9e-4415-a743-fc6a631877bb"
download(url3, "donnee-data.gouv-2024-geographie2025-produit-le2025-06-04.csv.gz")

Download successful: donnee-data.gouv-2024-geographie2025-produit-le2025-06-04.csv.gz


In [ ]:
# B - Read dataset 

In [34]:
crime = pd.read_csv("donnee-data.gouv-2024-geographie2025-produit-le2025-06-04.csv.gz", sep=";", compression="gzip", dtype={"CODGEO_2025": str})
crime.head()

,CODGEO_2025,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,est_diffuse,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,complement_info_nombre,complement_info_taux
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,"0,0000000",diff,767,2016,348,2016,NaN,NaN
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,ndiff,767,2016,348,2016,"1,3620690","0,9598386"
2,01001,2016,Violences sexuelles,Victime,0.0,"0,0000000",diff,767,2016,348,2016,NaN,NaN
3,01001,2016,Vols avec armes,Infraction,0.0,"0,0000000",diff,767,2016,348,2016,NaN,NaN
4,01001,2016,Vols violents sans arme,Infraction,0.0,"0,0000000",diff,767,2016,348,2016,NaN,NaN


In [35]:
crime.info()

<class 'pandas.DataFrame'>
RangeIndex: 4714200 entries, 0 to 4714199
Data columns (total 13 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   CODGEO_2025             str    
 1   annee                   int64  
 2   indicateur              str    
 3   unite_de_compte         str    
 4   nombre                  float64
 5   taux_pour_mille         str    
 6   est_diffuse             str    
 7   insee_pop               int64  
 8   insee_pop_millesime     int64  
 9   insee_log               int64  
 10  insee_log_millesime     int64  
 11  complement_info_nombre  str    
 12  complement_info_taux    str    
dtypes: float64(1), int64(5), str(7)
memory usage: 467.6 MB


In [ ]:
# C - Clean dataset 

In [ ]:
#C.1 - crime - Keep only needed columns

In [354]:
cols_to_drop = [
    "est_diffuse",
    "insee_pop_millesime",
    "insee_log",
    "insee_log_millesime",
    "complement_info_nombre",
    "complement_info_taux"
]
crime.drop(
    columns=cols_to_drop,
    inplace=True
)
crime.head()

,CODGEO_2025,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,insee_pop
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,"0,0000000",767
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,767
2,01001,2016,Violences sexuelles,Victime,0.0,"0,0000000",767
3,01001,2016,Vols avec armes,Infraction,0.0,"0,0000000",767
4,01001,2016,Vols violents sans arme,Infraction,0.0,"0,0000000",767


In [ ]:
#C.2 - crime - Renale columns

In [355]:
rename_dict = {
    "CODGEO_2025":"insee_commune_id",
    "indicateur":"crime_type",
    "insee_pop":"insee_pop_ref"
}
crime.rename(
    columns=rename_dict,
    inplace=True
)
crime.head()

,insee_commune_id,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,"0,0000000",767
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,767
2,01001,2016,Violences sexuelles,Victime,0.0,"0,0000000",767
3,01001,2016,Vols avec armes,Infraction,0.0,"0,0000000",767
4,01001,2016,Vols violents sans arme,Infraction,0.0,"0,0000000",767


In [ ]:
#C.3 - crime - Perform needed cast

In [356]:
crime.dtypes

insee_commune_id        str
annee                 int64
crime_type              str
unite_de_compte         str
nombre              float64
taux_pour_mille         str
insee_pop_ref         int64
dtype: object

In [357]:
crime["taux_pour_mille"] = pd.to_numeric(crime["taux_pour_mille"].str.replace(",", ".", regex=False), errors="coerce")
        # errors="coerce" ==> when conversion fails, change value in NaN
crime.head()

,insee_commune_id,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,0.0,767
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,767
2,01001,2016,Violences sexuelles,Victime,0.0,0.0,767
3,01001,2016,Vols avec armes,Infraction,0.0,0.0,767
4,01001,2016,Vols violents sans arme,Infraction,0.0,0.0,767


In [358]:
crime.dtypes

insee_commune_id        str
annee                 int64
crime_type              str
unite_de_compte         str
nombre              float64
taux_pour_mille     float64
insee_pop_ref         int64
dtype: object

In [ ]:
#C.4 - crime - Check null values

In [360]:
crime.info()

<class 'pandas.DataFrame'>
RangeIndex: 4714200 entries, 0 to 4714199
Data columns (total 7 columns):
 #   Column            Dtype  
---  ------            -----  
 0   insee_commune_id  str    
 1   annee             int64  
 2   crime_type        str    
 3   unite_de_compte   str    
 4   nombre            float64
 5   taux_pour_mille   float64
 6   insee_pop_ref     int64  
dtypes: float64(2), int64(2), str(3)
memory usage: 251.8 MB


In [361]:
crime.isna().sum()

insee_commune_id          0
annee                     0
crime_type                0
unite_de_compte           0
nombre              2174414
taux_pour_mille     2175177
insee_pop_ref             0
dtype: int64

In [362]:
crime["data_available"] = ~crime["nombre"].isna()

In [364]:
crime.head()

,insee_commune_id,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref,data_available
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,0.0,767,True
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,767,False
2,01001,2016,Violences sexuelles,Victime,0.0,0.0,767,True
3,01001,2016,Vols avec armes,Infraction,0.0,0.0,767,True
4,01001,2016,Vols violents sans arme,Infraction,0.0,0.0,767,True


In [ ]:
#count non zero and non null number of crimes

In [365]:
((crime["nombre"] != 0) & (crime["nombre"].notna())).sum()

np.int64(387422)

In [ ]:
#create another table with only the non zero, non null crime occurences

In [366]:
known_crimes = crime[crime["nombre"] > 0].copy()
known_crimes.info()

<class 'pandas.DataFrame'>
Index: 387422 entries, 30 to 4714199
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   insee_commune_id  387422 non-null  str    
 1   annee             387422 non-null  int64  
 2   crime_type        387422 non-null  str    
 3   unite_de_compte   387422 non-null  str    
 4   nombre            387422 non-null  float64
 5   taux_pour_mille   387422 non-null  float64
 6   insee_pop_ref     387422 non-null  int64  
 7   data_available    387422 non-null  bool   
dtypes: bool(1), float64(2), int64(2), str(3)
memory usage: 24.0 MB


In [ ]:
#C.5 - known_crimes & insee_to_commune - Flag communes with empty crime data in insee_to_commune table

In [367]:
insee_with_data = set(known_crimes["insee_commune_id"])
insee_to_commune["has_crime_data"] = insee_to_commune["insee_commune_id"].isin(insee_with_data)
insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True
4,01006,AMBLEON,Ambléon,False


In [ ]:
# Flag communes with empty revenu data in insee_to_commune table

In [368]:
insee_with_data2 = set(iris_to_commune["insee_commune_id"])
insee_to_commune["has_revenu_data"] = insee_to_commune["insee_commune_id"].isin(insee_with_data2)
insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_revenu_data
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False,False
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False,False
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True,True
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True,False
4,01006,AMBLEON,Ambléon,False,False


In [ ]:
#C.6 Join commune_name into known_crimes table

In [371]:
known_crimes.head(30)

,insee_commune_id,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref,data_available
30,01004,2016,Violences physiques intrafamiliales,Victime,28.0,1.988495,14081,True
31,01004,2016,Violences physiques hors cadre familial,Victime,53.0,3.763937,14081,True
32,01004,2016,Violences sexuelles,Victime,13.0,0.923230,14081,True
35,01004,2016,Vols sans violence contre des personnes,Victime entendue,133.0,9.445352,14081,True
36,01004,2016,Cambriolages de logement,Infraction,72.0,10.103681,14081,True
37,01004,2016,Vols de véhicule,Véhicule,49.0,3.479866,14081,True
38,01004,2016,Vols dans les véhicules,Véhicule,46.0,3.266813,14081,True
39,01004,2016,Vols d'accessoires sur véhicules,Véhicule,22.0,1.562389,14081,True
40,01004,2016,Destructions et dégradations volontaires,Infraction,116.0,8.238051,14081,True
41,01004,2016,Usage de stupéfiants,Mis en cause,29.0,2.059513,14081,True


In [372]:
commune_map = (
    insee_to_commune
    .drop_duplicates(subset="insee_commune_id")
    .set_index("insee_commune_id")["commune_name"]
)

pos = known_crimes.columns.get_loc("insee_commune_id")

known_crimes.insert(
    loc=pos + 1,
    column="commune_name",
    value=known_crimes["insee_commune_id"].map(commune_map)
)

known_crimes.head(30)

,insee_commune_id,commune_name,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref,data_available
30,01004,Ambérieu-en-Bugey,2016,Violences physiques intrafamiliales,Victime,28.0,1.988495,14081,True
31,01004,Ambérieu-en-Bugey,2016,Violences physiques hors cadre familial,Victime,53.0,3.763937,14081,True
32,01004,Ambérieu-en-Bugey,2016,Violences sexuelles,Victime,13.0,0.923230,14081,True
35,01004,Ambérieu-en-Bugey,2016,Vols sans violence contre des personnes,Victime entendue,133.0,9.445352,14081,True
36,01004,Ambérieu-en-Bugey,2016,Cambriolages de logement,Infraction,72.0,10.103681,14081,True
37,01004,Ambérieu-en-Bugey,2016,Vols de véhicule,Véhicule,49.0,3.479866,14081,True
38,01004,Ambérieu-en-Bugey,2016,Vols dans les véhicules,Véhicule,46.0,3.266813,14081,True
39,01004,Ambérieu-en-Bugey,2016,Vols d'accessoires sur véhicules,Véhicule,22.0,1.562389,14081,True
40,01004,Ambérieu-en-Bugey,2016,Destructions et dégradations volontaires,Infraction,116.0,8.238051,14081,True
41,01004,Ambérieu-en-Bugey,2016,Usage de stupéfiants,Mis en cause,29.0,2.059513,14081,True


In [ ]:
# add commune_name and place_in_commune name into disposable_income_2021 table

In [374]:
iris_to_commune.head()

,iris_id,insee_commune_id,commune_name,place_in_commune,has_income_data
0,010040101,01004,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,True
1,010040102,01004,Ambérieu-en-Bugey,Longeray-Gare,True
2,010040201,01004,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,True
3,010040202,01004,Ambérieu-en-Bugey,Tiret-Les Allymes,True
4,010330102,01033,Valserhône,Centre Ville,True


In [375]:
disposable_income_2021.head()

,iris_id,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_revenus_patrimoine_pct,part_prestations_sociales_pct,part_prestations_familiales_pct,part_minima_sociaux_pct,part_prestations_logement_pct,part_impots_pct
0,010040101,19.0,14990.0,20350.0,26140.0,11620.0,32060.0,70.8,26.9,6.2,8.6,3.3,3.8,1.5,-12.5
1,010040102,25.0,13880.0,18570.0,24760.0,10580.0,31130.0,70.6,24.9,5.8,11.1,3.7,5.1,2.3,-12.4
2,010040201,19.0,15190.0,20700.0,27180.0,11400.0,34450.0,72.5,27.2,6.4,7.7,2.8,3.3,1.6,-13.8
3,010040202,8.0,19600.0,25230.0,33170.0,14810.0,41230.0,73.3,23.8,16.2,4.0,1.8,1.5,0.7,-17.3
4,010330102,24.0,14050.0,20420.0,29640.0,9410.0,42390.0,78.9,23.7,5.2,5.3,1.5,2.5,1.3,-13.1


In [381]:
iris_map = (
    iris_to_commune
    .drop_duplicates(subset="iris_id")
    .set_index("iris_id")[["commune_name", "place_in_commune"]]
)

# Add the columns first
disposable_income_2021 = disposable_income_2021.join(
    iris_map,
    on="iris_id"
)

In [382]:
# Then move them
disposable_income_2021 = move_columns(
    disposable_income_2021,
    ["commune_name", "place_in_commune"],
    after="iris_id"
)

disposable_income_2021.head(30)

,iris_id,commune_name,place_in_commune,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_revenus_patrimoine_pct,part_prestations_sociales_pct,part_prestations_familiales_pct,part_minima_sociaux_pct,part_prestations_logement_pct,part_impots_pct
0,010040101,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,19.0,14990.0,20350.0,26140.0,11620.0,32060.0,70.8,26.9,6.2,8.6,3.3,3.8,1.5,-12.5
1,010040102,Ambérieu-en-Bugey,Longeray-Gare,25.0,13880.0,18570.0,24760.0,10580.0,31130.0,70.6,24.9,5.8,11.1,3.7,5.1,2.3,-12.4
2,010040201,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,19.0,15190.0,20700.0,27180.0,11400.0,34450.0,72.5,27.2,6.4,7.7,2.8,3.3,1.6,-13.8
3,010040202,Ambérieu-en-Bugey,Tiret-Les Allymes,8.0,19600.0,25230.0,33170.0,14810.0,41230.0,73.3,23.8,16.2,4.0,1.8,1.5,0.7,-17.3
4,010330102,Valserhône,Centre Ville,24.0,14050.0,20420.0,29640.0,9410.0,42390.0,78.9,23.7,5.2,5.3,1.5,2.5,1.3,-13.1
5,010330103,Valserhône,Lancrans-Coupy-Vanchy,17.0,16270.0,24060.0,35760.0,10700.0,49310.0,82.7,21.8,5.8,3.9,1.5,1.5,0.9,-14.2
6,010330201,Valserhône,Arc Vouvray-Gare-Châtillon,15.0,17940.0,25980.0,36690.0,11760.0,53670.0,79.2,26.4,6.0,3.2,1.2,1.3,0.7,-14.8
7,010330202,Valserhône,Plateau de Musinens,26.0,13750.0,19630.0,28050.0,10290.0,40430.0,70.2,30.4,3.7,8.0,2.6,3.7,1.7,-12.3
8,010330301,Valserhône,Arlod,19.0,15590.0,22360.0,31880.0,11260.0,43590.0,83.8,19.1,3.0,5.8,2.8,1.9,1.1,-11.7
9,010330401,Valserhône,Châtillon-en-Michaille,8.0,20830.0,28590.0,39750.0,15290.0,54750.0,86.2,19.2,7.3,2.6,1.3,0.8,0.5,-15.3


In [ ]:
# add commune_name and place_in_commune name into declared_income_2021 table

In [383]:
# Add the columns first
declared_income_2021 = declared_income_2021.join(
    iris_map,
    on="iris_id"
)
# Then move them
declared_income_2021 = move_columns(
    declared_income_2021,
    ["commune_name", "place_in_commune"],
    after="iris_id"
)

declared_income_2021.head(30)

,iris_id,commune_name,place_in_commune,menages_fiscaux_imposes_pct,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_autres_revenus_pct
0,010040101,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,43.0,29.0,12610.0,19330.0,26390.0,7760.0,33850.0,69.3,27.1,3.6
1,010040102,Ambérieu-en-Bugey,Longeray-Gare,42.0,39.0,9730.0,16830.0,24620.0,4680.0,33030.0,70.5,25.6,3.9
2,010040201,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,47.0,29.0,12220.0,19940.0,27650.0,6500.0,37220.0,69.3,26.7,4.0
3,010040202,Ambérieu-en-Bugey,Tiret-Les Allymes,62.0,14.0,18350.0,25560.0,35010.0,11960.0,45520.0,65.5,21.8,12.7
4,010330102,Valserhône,Centre Ville,42.0,31.0,11280.0,19870.0,30050.0,5250.0,44820.0,74.1,23.0,2.9
5,010330103,Valserhône,Lancrans-Coupy-Vanchy,49.0,23.0,14480.0,24250.0,37150.0,7390.0,52060.0,76.1,20.6,3.3
6,010330201,Valserhône,Arc Vouvray-Gare-Châtillon,50.0,19.0,16680.0,26260.0,38330.0,8940.0,57920.0,71.9,24.6,3.5
7,010330202,Valserhône,Plateau de Musinens,40.0,36.0,10810.0,18380.0,28270.0,5630.0,42550.0,68.1,30.3,1.6
8,010330301,Valserhône,Arlod,39.0,27.0,13000.0,21760.0,32830.0,7590.0,45980.0,79.8,18.7,1.5
9,010330401,Valserhône,Châtillon-en-Michaille,53.0,12.0,20080.0,29300.0,41880.0,13120.0,58590.0,77.5,17.6,4.9


In [ ]:
# Remove data_available column as all rows have data

In [373]:
known_crimes.drop(
    columns="data_available",
    inplace=True
)
known_crimes.head()

,insee_commune_id,commune_name,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref
30,01004,Ambérieu-en-Bugey,2016,Violences physiques intrafamiliales,Victime,28.0,1.988495,14081
31,01004,Ambérieu-en-Bugey,2016,Violences physiques hors cadre familial,Victime,53.0,3.763937,14081
32,01004,Ambérieu-en-Bugey,2016,Violences sexuelles,Victime,13.0,0.923230,14081
35,01004,Ambérieu-en-Bugey,2016,Vols sans violence contre des personnes,Victime entendue,133.0,9.445352,14081
36,01004,Ambérieu-en-Bugey,2016,Cambriolages de logement,Infraction,72.0,10.103681,14081


In [ ]:
#D - Save all cleaned tables in csv formats in current folder

In [384]:
known_crimes.to_csv("known_crimes_clean.csv", index=False)
crime.to_csv("crime_clean.csv", index=False)
declared_income_2021.to_csv("declared_income_2021_clean.csv", index=False)
disposable_income_2021.to_csv("disposable_income_2021_clean.csv", index=False)
insee_to_commune.to_csv("insee_to_commune_clean.csv", index=False)
iris_to_commune.to_csv("iris_to_commune_clean.csv", index=False)

In [1]:
#--------------------------------------------------------------
# II.b - CRIME analysis / Analyse d'EFFECTIF POLICE MUNICIPALE
#--------------------------------------------------------------

In [385]:
# REALISATION ULTERIEURE

In [ ]:
#sources: https://www.data.gouv.fr/datasets/police-municipale-effectifs-par-commune

In [359]:
#=========================================================================
# III - POLLUTION
#=========================================================================

In [ ]:
#-------------------------------------------------------------------------
# Quality of air analysis / Analyse de la qualite de l'air - source ATMO
#-------------------------------------------------------------------------

In [ ]:
# A - Download dataset - just index of the cities as a first step

In [45]:
url4 = "https://api.atmo-aura.fr/api/v1/communes?api_token=1cb18801f8f446e0a5a02136482aa85a"
response = requests.get(url4)
out = response.json()

In [ ]:
# B - Read index of cities dataset 

In [46]:
# Transform json in DataFrame
communes = pd.json_normalize(out["data"])
communes.head()

,code_insee,nom,epci_code,epci_nom,departement_code,departement_nom,region_code,region_nom,links.indices
0,74206,Orcier,200067551,CA Thonon Agglomération,74,Haute-Savoie,84,Auvergne-Rhône-Alpes,https://api.atmo-aura.fr/api/v1/communes/74206...
1,26370,Vercoiran,200068229,CC des Baronnies en Drôme Provençale,26,Drôme,84,Auvergne-Rhône-Alpes,https://api.atmo-aura.fr/api/v1/communes/26370...
2,26009,Andancette,200040491,CC Porte de Dromardèche,26,Drôme,84,Auvergne-Rhône-Alpes,https://api.atmo-aura.fr/api/v1/communes/26009...
3,03223,Saint-Christophe-en-Bourbonnais,240300491,CC du Pays de Lapalisse,03,Allier,84,Auvergne-Rhône-Alpes,https://api.atmo-aura.fr/api/v1/communes/03223...
4,42204,Saint-Bonnet-le-Château,200065886,CA Loire Forez Agglomération (LFA),42,Loire,84,Auvergne-Rhône-Alpes,https://api.atmo-aura.fr/api/v1/communes/42204...


In [ ]:
# C - Extract the API URL corresponding to Lyon city and download json ATMO indices

In [55]:
Lyon_details = duckdb.query("""
    SELECT nom, code_insee
    FROM communes
    WHERE nom = 'Lyon'
""").df()
display(Lyon_details)

,nom,code_insee
0,Lyon,69123


In [58]:
code_insee = Lyon_details.loc[0,"code_insee"]
url5 = f"https://api.atmo-aura.fr/api/v1/communes/{code_insee}/indices/atmo?api_token=1cb18801f8f446e0a5a02136482aa85a"
url5

'https://api.atmo-aura.fr/api/v1/communes/69123/indices/atmo?api_token=1cb18801f8f446e0a5a02136482aa85a'

In [65]:
response2 = requests.get(url5)
out2 = response2.json()
# Transform json in DataFrame
air_pollution = pd.json_normalize(out2["data"])
air_pollution.head()

,echeance,date_echeance,indice,qualificatif,couleur_html,date_calcul,commune_insee,commune_nom,type_valeur,sous_indices,polluants_majoritaires
0,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"[{'polluant_nom': 'SO2', 'concentration': 299....",[PM2.5]
1,0,2020-12-02,2,Moyen,#50ccaa,2020-12-02,69123,Lyon,prévision,"[{'polluant_nom': 'SO2', 'concentration': 2.9,...","[NO2, PM2.5]"
2,1,2020-12-03,2,Moyen,#50ccaa,2020-12-02,69123,Lyon,prévision,"[{'polluant_nom': 'SO2', 'concentration': 8.17...","[NO2, PM10, PM2.5]"
3,2,2020-12-04,2,Moyen,#50ccaa,2020-12-02,69123,Lyon,prévision,"[{'polluant_nom': 'SO2', 'concentration': 6.22...","[O3, NO2, PM2.5]"
4,-1,2020-12-06,4,Mauvais,#ff5050,2020-12-07,69123,Lyon,réelle,"[{'polluant_nom': 'PM2.5', 'concentration': 29...",[PM2.5]


In [66]:
# Extract sub levels of info
# 1. Explode the list of dictionaries
air_pollution = air_pollution.explode("sous_indices").reset_index(drop=True)
air_pollution.head()

,echeance,date_echeance,indice,qualificatif,couleur_html,date_calcul,commune_insee,commune_nom,type_valeur,sous_indices,polluants_majoritaires
0,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'SO2', 'concentration': 299.0...",[PM2.5]
1,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'O3', 'concentration': 52.32,...",[PM2.5]
2,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'NO2', 'concentration': 53.92...",[PM2.5]
3,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'PM10', 'concentration': 29.6...",[PM2.5]
4,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'PM2.5', 'concentration': 25....",[PM2.5]


In [67]:
# Normalize nested dictionaries
pollution_sub = pd.json_normalize(air_pollution["sous_indices"]).add_prefix("sub_").reset_index(drop=True)
pollution_sub.head()

,sub_polluant_nom,sub_concentration,sub_indice
0,SO2,299.04,3
1,O3,52.32,2
2,NO2,53.92,2
3,PM10,29.65,2
4,PM2.5,25.98,4


In [68]:
# Concatenate safely
air_pollution = pd.concat([air_pollution, pollution_sub], axis=1)
air_pollution.head()

,echeance,date_echeance,indice,qualificatif,couleur_html,date_calcul,commune_insee,commune_nom,type_valeur,sous_indices,polluants_majoritaires,sub_polluant_nom,sub_concentration,sub_indice
0,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'SO2', 'concentration': 299.0...",[PM2.5],SO2,299.04,3
1,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'O3', 'concentration': 52.32,...",[PM2.5],O3,52.32,2
2,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'NO2', 'concentration': 53.92...",[PM2.5],NO2,53.92,2
3,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'PM10', 'concentration': 29.6...",[PM2.5],PM10,29.65,2
4,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'PM2.5', 'concentration': 25....",[PM2.5],PM2.5,25.98,4


In [69]:
air_pollution.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   echeance                5000 non-null   int64  
 1   date_echeance           5000 non-null   str    
 2   indice                  5000 non-null   int64  
 3   qualificatif            5000 non-null   str    
 4   couleur_html            5000 non-null   str    
 5   date_calcul             5000 non-null   str    
 6   commune_insee           5000 non-null   str    
 7   commune_nom             5000 non-null   str    
 8   type_valeur             5000 non-null   str    
 9   sous_indices            5000 non-null   object 
 10  polluants_majoritaires  5000 non-null   object 
 11  sub_polluant_nom        5000 non-null   str    
 12  sub_concentration       5000 non-null   float64
 13  sub_indice              5000 non-null   int64  
dtypes: float64(1), int64(3), object(2), str(8)
memory u

In [70]:
air_pollution.describe()

,echeance,indice,sub_concentration,sub_indice
count,5000.000000,5000.000000,5000.000000,5000.000000
mean,0.500000,2.507000,37.130686,1.661800
std,1.118146,0.701462,33.951581,0.717442
min,-1.000000,1.000000,0.230000,1.000000
25%,-0.250000,2.000000,10.175000,1.000000
50%,0.500000,2.000000,22.420000,2.000000
75%,1.250000,3.000000,62.230000,2.000000
max,2.000000,5.000000,299.040000,5.000000


In [71]:
air_pollution

,echeance,date_echeance,indice,qualificatif,couleur_html,date_calcul,commune_insee,commune_nom,type_valeur,sous_indices,polluants_majoritaires,sub_polluant_nom,sub_concentration,sub_indice
0,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'SO2', 'concentration': 299.0...",[PM2.5],SO2,299.04,3
1,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'O3', 'concentration': 52.32,...",[PM2.5],O3,52.32,2
2,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'NO2', 'concentration': 53.92...",[PM2.5],NO2,53.92,2
3,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'PM10', 'concentration': 29.6...",[PM2.5],PM10,29.65,2
4,-1,2020-12-01,4,Mauvais,#ff5050,2020-12-02,69123,Lyon,réelle,"{'polluant_nom': 'PM2.5', 'concentration': 25....",[PM2.5],PM2.5,25.98,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,2,2021-08-18,2,Moyen,#50ccaa,2021-08-16,69123,Lyon,prévision,"{'polluant_nom': 'NO2', 'concentration': 44.05...","[NO2, O3]",NO2,44.05,2
4996,2,2021-08-18,2,Moyen,#50ccaa,2021-08-16,69123,Lyon,prévision,"{'polluant_nom': 'O3', 'concentration': 81.44,...","[NO2, O3]",O3,81.44,2
4997,2,2021-08-18,2,Moyen,#50ccaa,2021-08-16,69123,Lyon,prévision,"{'polluant_nom': 'SO2', 'concentration': 0.71,...","[NO2, O3]",SO2,0.71,1
4998,2,2021-08-18,2,Moyen,#50ccaa,2021-08-16,69123,Lyon,prévision,"{'polluant_nom': 'PM10', 'concentration': 15.0...","[NO2, O3]",PM10,15.02,1
